In [ ]:
import probablepeople as pp 
import os
import sys
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.functions import dense_rank, desc,when
from pyspark.sql.functions import dense_rank, desc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [ ]:
import os
import sys

# 1. Point directly to your installed JDK 17 path
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]

# Ensure PySpark uses the active Python interpreter
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 2. Now import and start Spark


spark = SparkSession.builder \
    .appName("pyspark_ylep") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "24") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.autoBroadcastJoinThreshold", "64MB") \
    .getOrCreate()
print("Spark initialized successfully!")

In [ ]:
df_read_business_data = spark.read.json("D:\\A_Data_engineer\\Project_Yelp\\Yelp JSON\\yelp_academic_dataset_business.json")

In [ ]:
invalid_placeholders=["", "N/A", "null", "None","Confidential"]

f_new_address_1 = df_read_business_data.withColumn( "new_address" , F.when( F.trim(F.col("address")).isin(invalid_placeholders)
                                                                             , "unknow address" ) .otherwise(F.regexp_replace(F.lower(F.col("address")) ,r"\bstreet\b","st") ) )


In [ ]:

# Dictionary of old_word: new_word
replacements = {
                r"-davidson metropolitan government \(balance\)": "",    
                r"\s*city \(balance\)": "",    
                r"\s*\(meridian\)": "", 
                r"\s*\(walsingham\)": "",
                r"\s*\(northeast philadelphia\)": "",
                r"\bfl\b\s*":"",
                r"\s*/.*$": "",
                r"\bn\b\.\s*":"north",
                r"\bst\b\.\s*":"saint",
                r"\bmt\b\.\s*":"mount juliet",
                r"\bhts\b\.\s*":"heights",
                r"\be\b\.\s*":"east",
                r"\bw\b\.\s*":"west",
                r"\bkop\b\s*":"king of prussia", 
                r"\btwp\b\s*":"township", 
                r"\bnj\b\s*":"new jersey", 
                r"\bphilly\b\s*":"philadelphia",
                ",": "",
                "&": "and",
                }

# Chain regexp_replace for each pair
expr = F.lower(F.col("city"))
for old, new in replacements.items(): 
    expr = F.regexp_replace(expr, old, new)

df_city = df_read_business_data.withColumn("cleaned_city", expr)



In [ ]:
combined_regex = r"^\d{5}(-\d{4})?$|^(?i)[ABCEGHJ-NPRSTVXY]\d[ABCEGHJ-NPRSTV-Z][ ]?\d[ABCEGHJ-NPRSTV-Z]\d$"
df_state_1 = df_read_business_data.withColumn("new_state", when( F.col("state").isNull(),"unknown state")\
                                           .when(F.upper(F.col("state")).rlike( combined_regex ),"unknown state")\
                                           .otherwise(F.upper(F.col("state")))
                                              )


In [ ]:
df_state_teritory = spark.read \
    .option("header", "true") \
    .option("sep", ",") \
    .csv("D:\\A_Data_engineer\\Project_Yelp\\US_GeoCode.csv")
dict_state =dict()
for i,j in df_state_teritory.select("state&teritory","latitude").collect():
   dict_state[i]=j


from itertools import chain

col_latitude = F.col("latitude").cast("float")

valid_condition= col_latitude.isNotNull() & col_latitude.between(-90.0,90.0) 

map_expr = F.create_map([F.lit(x) for x in chain(*dict_state.items())])

df_latitude =  df_read_business_data.withColumn("new_latitude" ,when(valid_condition,col_latitude).otherwise( map_expr.getItem(F.col("state")) ))

In [ ]:
dict_longitude =dict()
for i,j in df_state_teritory.select("state&teritory","longitude").collect():
   dict_longitude[i]= j

In [ ]:
from itertools import chain

col_longitude= F.col("longitude").cast("float")

valid_condition= col_longitude.isNotNull() & col_longitude.between(-180.0,180.0) 

map_expr = F.create_map( [F.lit(x) for x in chain(*dict_longitude.items())] )

df_latitude =  df_read_business_data.withColumn("new_longitude" ,when(valid_condition,col_longitude).otherwise( map_expr.getItem(F.col("state"))))

In [ ]:
stars = F.col("stars").try_cast("float")
review_count = F.col("review_count").try_cast("int") 
is_open = F.col("is_open").try_cast("int")
df_valid_review_number = df_read_business_data.withColumn("new_stars",F.when( ( stars >=0 ) | ( stars <=5 ) ,stars).otherwise(0) )\
                                              .withColumn("new_review_count",F.when( ( review_count >=0 ),review_count).otherwise(0) )\
                                              .withColumn("new_is_open",F.when(  is_open == 1 ,1 ).otherwise(0) )

In [ ]:
def clean_address(text):
    wifi=""
    alcohol=""
    RestaurantsDelivery=""
    RestaurantsTakeOut=""
    NoiseLevel=""
    GoodForKids=""
    RestaurantsAttire=""
    RestaurantsReservations=""
    OutdoorSeating=""
    Open24Hours=""
    RestaurantsPriceRange=""
    BusinessParking=""
    set_attributes=set()
    split_string = str(text).split(",")
    for values_of_split_string in  split_string :
        set_attributes.add(values_of_split_string)
    for individual_attribute_ in set_attributes:
        attribute=individual_attribute_.lower()
        if ("wifi") in attribute :

            if ("paid") in attribute:
               wifi="Paid"
            elif ("free") in attribute:
               wifi="Free"
            else:
               wifi= "Not Available"

        if ("alcohol") in attribute :

            if ("beer") in attribute:
               alcohol="Beer And Wine"
            elif ("bar") in attribute:
               alcohol="Full Bar"
            else:
               alcohol= "Not Available"

        if ("delivery") in attribute :

            if ("true") in attribute:
                    RestaurantsDelivery="Available"
            else:
                    RestaurantsDelivery= "Not Available"  

        if ("takeout") in attribute :

            if ("true") in attribute:
                    RestaurantsTakeOut="Available"
            else:
                    RestaurantsTakeOut= "Not Available"  

        if ("noise") in attribute :

            if ("loud") in attribute:
               NoiseLevel="loud"

            elif ("average") in attribute:
               NoiseLevel="average"

            elif ("very_loud") in attribute:
               NoiseLevel="very_loud"

            elif ("quiet") in attribute:
               NoiseLevel="quiet"

            else:
               NoiseLevel="Not Specified"                    


        if ("goodforkids") in attribute :

            if ("true") in attribute:
               GoodForKids="Suitable"

            else:
               GoodForKids="Not Suitable"                   


        if ("attire") in attribute :

            if ("dressy") in attribute:
               RestaurantsAttire="dressy"

            elif ("formal") in attribute:
               RestaurantsAttire="formal"

            else:
               RestaurantsAttire="casual"                   

        
        if ("reservations") in attribute :

            if ("true") in attribute:
               RestaurantsReservations="accepted"

            else:
               RestaurantsReservations="Not accepted"     

        if ("outdoorseating") in attribute :

            if ("true") in attribute:
               OutdoorSeating="Available"

            else:
               OutdoorSeating="Not Available"   

        if ("open24hours") in attribute :

            if ("true") in attribute:
               Open24Hours="Open 24 Hours"

            else:
               Open24Hours="Standard Hours"   


        if ("pricerange") in attribute :

            if ("1") in attribute:
               RestaurantsPriceRange="Inexpensive"

            elif ("2") in attribute:
               RestaurantsPriceRange="Moderate"

            elif ("3") in attribute:
               RestaurantsPriceRange="Pricey"

            elif ("4") in attribute:
               RestaurantsPriceRange="Ultra"
            else:
               RestaurantsPriceRange="Not Specified" 


        if ("businessparking") in attribute :

            if ("garage") in attribute:
                if ("true") in attribute:
                    BusinessParking="Available"

            else:
               BusinessParking="Not Available"                  

               

    if not wifi:
        wifi= "Not Available"   

    if not alcohol:
        alcohol= "Not Available"   

    if not RestaurantsDelivery:
        RestaurantsDelivery= "Not Available" 

    if not RestaurantsTakeOut:
        RestaurantsTakeOut= "Not Available"    

    if not NoiseLevel:
        NoiseLevel= "Not Specified"     

    if not GoodForKids:
        GoodForKids= "Not Suitable"   

    if not RestaurantsAttire:
        RestaurantsAttire="casual" 

    if not RestaurantsReservations:
        RestaurantsReservations="Not accepted" 

    if not OutdoorSeating:
        OutdoorSeating="Not Available"   

    if not Open24Hours:
        Open24Hours="Standard Hours"  

    if not RestaurantsPriceRange:
        RestaurantsPriceRange="Not Specified" 

    if not BusinessParking:
        BusinessParking="Not Available" 

    return wifi,alcohol,RestaurantsDelivery,RestaurantsTakeOut,NoiseLevel,GoodForKids,RestaurantsAttire,RestaurantsReservations,OutdoorSeating,BusinessParking,Open24Hours,RestaurantsPriceRange


# define the schema for 
schema = StructType([
    StructField("WiFi", StringType(), True),
    StructField("Alcohol", StringType(), True),
    StructField("BusinessDelivery", StringType(), True),
    StructField("RestaurantsTakeOut", StringType(), True),
    StructField("NoiseLevel", StringType(), True),
    StructField("GoodForKids", StringType(), True),
    StructField("RestaurantsAttire", StringType(), True),
    StructField("RestaurantsReservations", StringType(), True),
    StructField("OutdoorSeating", StringType(), True),
    StructField("BusinessParking", StringType(), True),
    StructField("Open24Hours", StringType(), True),
    StructField("RestaurantsPriceRange", StringType(), True)
])            
clean_address_udf = F.udf(clean_address, schema)


df_elite = (
    df_read_business_data.withColumn("new_eilte", clean_address_udf(F.col("attributes")))
      .withColumn("new_wifi", F.col("new_eilte.WiFi"))
      .withColumn("new_alcohol", F.col("new_eilte.Alcohol"))
      .withColumn("new_BusinessDelivery", F.col("new_eilte.BusinessDelivery") )
      .withColumn("new_RestaurantsTakeOut", F.col("new_eilte.RestaurantsTakeOut") )
      .withColumn("new_NoiseLevel", F.col("new_eilte.NoiseLevel") )
      .withColumn("new_GoodForKids", F.col("new_eilte.GoodForKids") )
      .withColumn("new_RestaurantsAttire", F.col("new_eilte.RestaurantsAttire"))
      .withColumn("new_RestaurantsReservations", F.col("new_eilte.RestaurantsReservations"))
      .withColumn("new_OutdoorSeating", F.col("new_eilte.OutdoorSeating") )
      .withColumn("new_BusinessParking", F.col("new_eilte.BusinessParking") )
      .withColumn("new_Open24Hours", F.col("new_eilte.Open24Hours") )
      .withColumn("new_RestaurantsPriceRange", F.col("new_eilte.RestaurantsPriceRange") )      
      .drop("new_eilte",F.col("attributes"))
)  
df_elite.show(n=1000)